<img src="../logo.png" alt="Vegeta" width="240">

# Four races — the reference 5-inch quad round the hangar, the industrial zone, the farm and the quarry

Four race environments for MuJoCo (`races/`): two technical courses (a three-floor **hangar**, an **industrial zone**
between large buildings) and two open ones (a **farm**, a **quarry**). Each is an ordered list of **key zones** —
boxes the racer must pass through, all of them, in order — with checkpoints, a starting grid and a finish. One
reference aircraft: a conventional 5-inch FPV racing quad (2207 1750 KV on 6S, 5x4.3 tri-blades, a 6S 1300 mAh pack,
571 g from its component budget). One autopilot, one set of rules.

**The MuJoCo races are workflow chapters** (book **races**, `wf R1`–`R4`: `side_labs/races/race.py` runs the lap in
MuJoCo and films it — chase, onboard, overview — into a run directory). This notebook **takes** those runs through
its Yanus gate: when they are on file it shows their statistics, manoeuvre metrics and videos; when they are not, it
**runs the missing chapters itself** in MuJoCo (the notebook is `yanus_auto`: the gate drives `wf R1`–`R4` in
screening quality — short, small videos — before showing them; `YANUS_AUTO=0` turns that off, `YANUS_AUTO_MINUTES`
caps each run). A course whose run failed falls back to the **mock flyer** (a point mass on the same schedule, the
same referee, the same drive table for the energy: the shape of the real result and a lower bound of its lap times),
with the failure named. Camera videos need an OpenGL context (`MUJOCO_GL`, `egl` by default); without one the chapter
films the replay instead and says so.

In [ ]:
DEBUG = True
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import os, sys, time, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
sys.path.insert(0, "../..")
from races import courses, mock, refquad, replay, rules, path as rpath
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
from vegeta import yanus
GATE = yanus.enter("50_four_races")              # takes: the four race runs (wf R1–R4). Auto mode is on for this notebook: the missing ones are run now, in MuJoCo, screening quality (short videos) — a few minutes on the first run
display(GATE.table())
if getattr(GATE, "auto", None):
    print("auto mode ran:"); [print("  ", st) for st in GATE.auto]
T0 = time.time()
OUT = Path("../../races/runs/notebook"); OUT.mkdir(parents=True, exist_ok=True)
from races import metrics, video
KEYS = {slug: f"races/{slug}" for slug in courses.COURSES}
RUNS = {slug: GATE.inputs.get(k) for slug, k in KEYS.items()}                              # the summary.json of each usable run, else None
RUN_DIRS = {slug: next((Path("../..") / r.selection for r in GATE.rows if r.kind == "run" and r.key == k and r.usable), None) for slug, k in KEYS.items()}
print({slug: ("on file: " + str(RUN_DIRS[slug])) if RUNS[slug] else "NOT RUN — the mock below; run `wf R%d` or set YANUS_AUTO=1" % (i + 1) for i, slug in enumerate(KEYS)})

## 1. The reference racer

The component budget and what the existing propulsion model says about it. The drive table is the Boreas
blade-element table of this exact drive (made by `mini_raptor.drive_table`), the frame's drag area an assumption.

In [ ]:
display(pd.Series(refquad.BUDGET, name="mass [g]").to_frame().T)
display(pd.Series(refquad.summary(), name="reference quad").to_frame())
d = refquad.drive()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(d.V, d.T_max); ax[0].set(xlabel="axial inflow [m/s]", ylabel="full-throttle thrust per motor [N]", title="the 6S 5x4.3 2207 drive"); ax[0].grid(alpha=0.3)
ax[1].plot(d.T, [d.power(t, 0) for t in d.T], label="static"); ax[1].plot(d.T, [d.power(t, 20) for t in d.T], label="20 m/s inflow"); ax[1].set(xlabel="thrust per motor [N]", ylabel="electrical [W]", xlim=(0, 10), ylim=(0, 400)); ax[1].grid(alpha=0.3); ax[1].legend()
plt.show()

## 2. The rules

The same for every course (`races.rules`): zones in order; a later zone first is a missed gate and the racer must go
back; checkpoints are reset points; any contact once airborne is a collision — reset to the last checkpoint with a
5 s penalty (the default), or counted, or race over; more than five collisions or 300 s is a DNF; the lap is complete
when the finish is entered after every other zone; the lap time runs from leaving the start zone to entering the
finish, penalties included.

In [ ]:
print(rules.__doc__)
display(pd.Series(rules.Rules().__dict__, name="defaults").drop("text").to_frame())

## 3. The courses

Each course: its map (plan and elevation) with the obstacles, the landmarks, the numbered zones (green start, blue
gates, orange checkpoints, red finish), the starting grid; the zones' clearance; the racing line's speed schedule
for the reference quad.

In [ ]:
COURSES = {slug: fn() for slug, fn in courses.COURSES.items()}
for slug, c in COURSES.items():
    display(Markdown(f"### {c.name} — {c.kind}, {c.length():.0f} m, {len(c.zones)} zones\n\n{c.description}.  \n*Recovery:* {c.recovery}.  \n*Parameters:* `{c.params}`"))
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [2, 1]})
    replay.overview(c, ax=axes[0], view="plan"); replay.overview(c, ax=axes[1], view="elevation"); plt.tight_layout(); plt.show()
    ck = c.clearance_check()
    display(pd.DataFrame({"zone": [z.name for z in c.zones], "kind": [z.kind for z in c.zones], "centre": [z.center for z in c.zones], "half size": [z.half for z in c.zones],
                          "nearest obstacle [m]": ck["nearest obstacle [m]"].values, "clear": ck["ok"].values}).set_index("zone"))
    sched = mock.schedule(c)
    print(f"schedule: {sched.time_estimate():.1f} s at up to {sched.v.max():.0f} m/s (lateral {sched.limits['a_lateral'] / 9.81:.1f} g, brake {sched.limits['a_brake'] / 9.81:.1f} g)")
print(f"§ 3 in {time.time() - T0:.0f} s")

## 4. The races

For each course: the MuJoCo run when it is on file (its statistics, its gate splits, its videos), else the mock
flyer — the flown path over the map, speed and power, the gate splits, the referee's statistics, and a short GIF
replay written to `races/runs/notebook/`.

In [ ]:
from IPython.display import Video
MOCK = {}
rows = []
for slug, c in COURSES.items():
    display(Markdown(f"### {c.name}"))
    S = RUNS[slug]
    if S and not S.get("stats"):                                          # a run on file whose lap left no statistics (the case failed): say so, fall back to the mock
        notes = {k: v.get("note") or v.get("status") for k, v in (S.get("cases") or {}).items()}
        display(Markdown(f"**Run on file without a lap** `{RUN_DIRS[slug]}`: {S.get('note') or 'the race case produced no statistics'} — case records: `{notes}`. Read its run.log; the mock follows."))
        RUNS[slug] = S = None
    if S:
        display(Markdown(f"**MuJoCo run on file** ({S.get('mode')}, quality {S.get('quality')}): `{RUN_DIRS[slug]}`"))
        display(pd.Series(S["stats"], name="referee").to_frame().T.T)
        display(pd.Series(S.get("gate_times") or {}, name="split [s]").to_frame().T)
        for cam, rel in (S.get("videos") or {}).items():
            f = RUN_DIRS[slug] / rel
            display(Markdown(f"*{cam} camera* — `{f}`"))
            if f.exists() and f.suffix == ".mp4":
                display(Video(str(f), embed=False, width=640))
            elif f.exists():
                display(Image(filename=str(f)))
        rows.append({"course": c.name, "mode": S.get("mode"), **{k: v for k, v in S["stats"].items() if k not in ("course", "aircraft", "mode")}})
        continue
    r = mock.run_mock(c)
    MOCK[slug] = r
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [2, 1]})
    replay.overview(c, r, ax=axes[0], view="plan"); replay.overview(c, r, ax=axes[1], view="elevation"); plt.tight_layout(); plt.show()
    replay.traces(r); plt.show()
    display(replay.splits(r).round(2).T)
    rows.append({"course": c.name, **{k: v for k, v in r["stats"].items() if k not in ("course", "aircraft")}})
    gif = replay.animate(c, r, OUT / f"{slug}_mock.gif", every=6, max_frames=90)
    print("replay:", gif)
RACES = pd.DataFrame(rows).set_index("course")
print(f"§ 4 in {time.time() - T0:.0f} s")

## 5. Results

In [ ]:
display(RACES.T)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
RACES["lap time [s]"].plot.bar(ax=ax[0], color="#3377dd", title="lap time [s] (mock, a lower bound)"); RACES["energy [Wh]"].plot.bar(ax=ax[1], color="#f29d1c", title="energy per lap [Wh] of 23 usable")
RACES["mean speed [m/s]"].plot.bar(ax=ax[2], color="#2a9d2a", title="mean speed [m/s]")
for a in ax: a.grid(alpha=0.3, axis="y"); a.set_xlabel("")
plt.tight_layout(); plt.show()
for slug in MOCK: display(Image(filename=str(OUT / f"{slug}_mock.gif")))

## 5b. Manoeuvrability

What the laps asked of the aircraft and what it could give (`races.metrics`): the linear envelope (peak acceleration
and deceleration along the path, peak lateral acceleration, the load factor the thrust carried), the turning (least
turn radius above 3 m/s, peak turn rate), the rotational envelope (peak roll / pitch / yaw rates and angular
accelerations — from the recorded body rates in a MuJoCo run; from the thrust vector's slew and the heading in the
mock, which has no attitude), the time to speed, the tilt used, the share of the lap near full thrust, and the
control quality (the tracking error to the racing line). Beside them the aircraft's **envelope** from its drive table
and an inertia estimate, and the **literature**: Foehn, Romero & Scaramuzza (Science Robotics 2021) plan time-optimal
racing with accelerations of 1–20 m/s² and body rates up to 10 rad/s on a platform of thrust-to-weight 3.3–4; UZH's
Swift (Kaufmann et al., Nature 2023) raced champions at over 100 km/h; typical FPV rate settings are 800–1200 °/s in
roll and pitch. "Peaks" are 99th percentiles: differentiated samples spike at the zone hand-overs.

In [ ]:
runs_for_metrics = {}
for slug, c in COURSES.items():
    if RUNS[slug] and RUN_DIRS[slug] and (RUN_DIRS[slug] / "cases" / "race" / "trajectory.npz").exists():
        runs_for_metrics[f"{c.name} (MuJoCo)"] = {"history": np.load(RUN_DIRS[slug] / "cases" / "race" / "trajectory.npz")["history"], "path": mock.schedule(c)}
    elif slug in MOCK:
        runs_for_metrics[f"{c.name} (mock)"] = MOCK[slug]
MAN = metrics.table(runs_for_metrics)
display(MAN.round(2))
display(pd.Series(metrics.envelope(), name="the reference quad's envelope").to_frame())
display(metrics.literature())
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, key in zip(ax, ("peak lateral acceleration [g]", "least turn radius above 3 m/s [m]", "peak load factor (thrust / weight carried)")):
    MAN.loc[key].drop("aircraft envelope", errors="ignore").astype(float).plot.bar(ax=a, color="#3377dd", title=key)
    if key in MAN.index and "aircraft envelope" in MAN.columns and np.isfinite(float(MAN.loc[key, "aircraft envelope"])): a.axhline(float(MAN.loc[key, "aircraft envelope"]), color="#d62828", ls="--", lw=1, label="envelope (70° tilt / static)"); a.legend(fontsize=8)
    a.grid(alpha=0.3, axis="y"); a.set_xlabel("")
plt.tight_layout(); plt.show()

## 6. Running it in MuJoCo

```
pip install mujoco                               # and ffmpeg on the PATH for the videos
wf R1                                            # the hangar: the lap and its three videos into side_labs/runs/races/hangar/<run>/
wf R3 --quality fine                             # the farm with full-length, full-size videos
YANUS_AUTO=1 jupyter nbconvert --execute 50_four_races.ipynb    # or: this notebook produces the missing runs itself (screening)
python -m races run --course farm --gif          # the thin CLI outside the run contract
```
Camera views need an OpenGL context for `mujoco.Renderer` (`MUJOCO_GL=egl` on a headless machine); without one the
chapter films the replay instead and says so in the case record.
`races.sim.render(course, result, out_dir, camera="onboard"|"chase"|"overview"|"grandstand")` makes frames from a
recorded run (an OpenGL context is needed). What to expect on the first real runs: the controller's gains
(`omega_pos`, `omega_att`, the 70° tilt limit) and the schedule's `safety` factor will need tuning per course; the
referee, the zones and the statistics are the ones above.

## 7. Findings (mock)

- All four courses are flyable by the reference quad at the published schedule with a metre or more of clearance at
  every zone; the mock completes each with no collision and no missed gate, and the rules behave (a forced collision
  resets to the last checkpoint with its penalty; `stop` ends the race).
- The technical courses are flown at 9–17 m/s mean and the open ones at 15–20 m/s; a lap costs 1–3 Wh of the pack's
  23, so a race of several laps is well inside the battery.
- The mock's lap times are what a perfectly tracking quad does on this schedule: a real controller lags, so expect
  longer laps and the occasional reset where the zones are tight (the hangar's stair openings, the barn doors, the
  quarry corridors).

In [ ]:
doc = {"schema": 1, "notebook": "50_four_races", "mode": {slug: ("mujoco run on file" if RUNS[slug] else "mock") for slug in COURSES}, "manoeuvre": MAN.to_dict(), "envelope": metrics.envelope(), "literature": metrics.LITERATURE, "aircraft": refquad.summary(), "budget_g": refquad.BUDGET, "rules": {k: v for k, v in rules.Rules().__dict__.items() if k != "text"},
       "courses": {slug: {"name": c.name, "kind": c.kind, "length_m": c.length(), "zones": [{"name": z.name, "kind": z.kind, "center": z.center, "half": z.half} for z in c.zones], "params": c.params, "description": c.description, "recovery": c.recovery} for slug, c in COURSES.items()},
       "races": {slug: ({"stats": RUNS[slug]["stats"], "gate times": RUNS[slug].get("gate_times"), "videos": RUNS[slug].get("videos"), "run": str(RUN_DIRS[slug])} if RUNS[slug] else
                        {"stats": {k: (None if isinstance(v, float) and not np.isfinite(v) else v) for k, v in MOCK[slug]["stats"].items()}, "gate times": MOCK[slug]["gate times"]}) for slug in COURSES}}
out = Path("../../designs/data/four_races.json"); out.write_text(json.dumps(doc, indent=1, default=float))
print("exported", out, f"{out.stat().st_size / 1e3:.0f} kB")
GATE.leave()
print(f"done in {time.time() - T0:.0f} s")